In [ ]:
import numpy as np
import xarray as xr

# ---- pixel runner (must be importable by workers) ----
def run_pixel_ts(T_arr, S_arr, Chl_arr, POC_arr, POM_arr, TPM_arr, time_index_array, model_cfg_path, init_state=None):
    """
    T_arr, S_arr, ... : 1D numpy arrays of length Ntime (float)
    time_index_array   : 1D numpy array of datetimes or floats (not used directly by pyfabm unless needed)
    model_cfg_path     : path to fabm.yaml (string)
    init_state         : 1D array length n_state_vars (optional)
    returns: np.ndarray shape (n_variables, Ntime)
    """
    import pyfabm
    # instantiate model inside function (important)
    model = pyfabm.Model(model_cfg_path)
    model.cell_thickness = 1.0
    # set any model-wide constants
    model.dependencies["seeding_rate"].value = 0.0
    model.dependencies["harvest_ratio"].value = 0.0
    model.dependencies["air_exposure"].value = 0.0

    # start model
    if not model.start():
        raise RuntimeError("pyfabm:start failed: " + pyfabm.getError())

    ntime = T_arr.shape[0]
    nvars = model.state.size  # should be 11 in your config
    out = np.zeros((nvars, ntime), dtype=float)

    # initial state
    if init_state is not None:
        model.state[:] = init_state
    else:
        # keep whatever defaults are in the model, or zero
        pass

    for i in range(ntime):
        # set environmental dependencies
        model.dependencies["temperature"].value = float(T_arr[i])
        model.dependencies["practical_salinity"].value = float(S_arr[i])
        # set food fields: findStateVariable(...) may be slower each loop; cache names if needed
        model.findStateVariable('Chl1/Chl').value = float(Chl_arr[i])
        model.findStateVariable('POC1/POC').value = float(POC_arr[i])
        model.findStateVariable('POM1/POM').value = float(POM_arr[i])
        model.findStateVariable('TPM1/TPM').value = float(TPM_arr[i])

        # compute rates and integrate for 1 day
        rates = model.getRates()
        model.state[:] += rates * 86400.0  # seconds -> per day

        out[:, i] = model.state[:]  # copy state snapshot

    return out


In [ ]:
# Suppose ds_T, ds_S, ds_Chl, ... are DataArrays with dims ('time','latitude','longitude')
# and chunking applied such that each block contains many lat/lon but full time

from dask.distributed import Client
client = Client()  # tune workers/threads per your machine

# wrapper function that apply_ufunc will call per-lat/lon point
def _apply_wrapper(T, S, Chl, POC, POM, TPM, times):
    return run_pixel_ts(T, S, Chl, POC, POM, TPM, times, model_cfg_path="/path/to/fabm.yaml")

# apply_ufunc
result = xr.apply_ufunc(
    _apply_wrapper,
    ds_T, ds_S, ds_Chl, ds_POC, ds_POM, ds_TPM,
    input_core_dims=[['time'], ['time'], ['time'], ['time'], ['time'], ['time']],
    output_core_dims=[['variable','time']],
    vectorize=False,
    dask='parallelized',
    output_dtypes=[float],
    output_sizes={'variable': 11, 'time': ds_T.sizes['time']}
)

# then reorder dims to ('time','latitude','longitude','variable') if you prefer
result = result.transpose('time', 'latitude', 'longitude', 'variable')
